In [1]:
%pip install pandas==2.2.3 streamlit==1.49.1

   ---------------------------------------- 0.0/11.5 MB ? eta -:--:--
   -------------------------- ------------- 7.6/11.5 MB 47.1 MB/s eta 0:00:01
   ---------------------------------------- 11.5/11.5 MB 45.0 MB/s eta 0:00:00
   ---------------------------------------- 0.0/10.0 MB ? eta -:--:--
   ---------------------------------------  10.0/10.0 MB 68.8 MB/s eta 0:00:01
   ---------------------------------------- 10.0/10.0 MB 48.2 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 7.36.0
    Uninstalling protobuf-7.36.0:
      Successfully uninstalled protobuf-7.36.0
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
  Attempting uninstall: streamlit
    Found existing installation: streamlit 1.37.1
    Uninstalling streamlit-1.37.1:
      Successfully uninstalled streamlit-1.37.1
Note: you may need to restart the kernel to use updated package

  You can safely remove it manually.


In [3]:
%%writefile app.py
from datetime import date, timedelta
from pathlib import Path

import pandas as pd
import streamlit as st


def load_people(source):
    """Load and validate the people CSV."""
    df = pd.read_csv(
        source,
        dtype=str,
        keep_default_na=False,
        encoding="utf-8-sig",
    )

    # Normalize capitalization and extra spaces in column names.
    df.columns = [
        " ".join(column.lstrip("\ufeff").split()).lower()
        for column in df.columns
    ]

    if df.columns.duplicated().any():
        raise ValueError("Duplicate column names found.")

    required = {
        "user id",
        "first name",
        "last name",
        "date of birth",
    }

    missing = required - set(df.columns)

    if missing:
        raise ValueError(
            "Missing required columns: " + ", ".join(sorted(missing))
        )

    if df.empty:
        raise ValueError("The CSV contains no people.")

    for column in df.columns:
        df[column] = df[column].str.strip()

    if df["user id"].eq("").any():
        raise ValueError("User IDs cannot be empty.")

    if df["user id"].duplicated().any():
        raise ValueError("User IDs must be unique.")

    birth_dates = pd.to_datetime(
        df["date of birth"],
        format="%Y-%m-%d",
        errors="coerce",
    )

    invalid = (
        birth_dates.isna()
        | (birth_dates > pd.Timestamp(date.today()))
    )

    if invalid.any():
        raise ValueError(
            f"{int(invalid.sum())} invalid or future birth date(s) found. "
            "Use YYYY-MM-DD."
        )

    df["birth_date"] = birth_dates.dt.date

    df["label"] = (
        df["first name"]
        + " "
        + df["last name"]
        + " — "
        + df["user id"]
    )

    return df


def age_on(born, on_date):
    """
    Calculate age in completed years.

    For February 29 births, age advances on March 1
    during non-leap years.
    """
    if on_date < born:
        raise ValueError("The selected date is before the birth date.")

    birthday_not_reached = (
        (on_date.month, on_date.day)
        < (born.month, born.day)
    )

    return on_date.year - born.year - int(birthday_not_reached)


def make_forecast(people, user_id, days=30, start=None):
    """
    Calculate daily ages for the selected person.

    Includes today plus the requested number of future days.
    This is an exact calculation, not a machine-learning forecast.
    """
    if (
        isinstance(days, bool)
        or not isinstance(days, int)
        or not 1 <= days <= 3650
    ):
        raise ValueError(
            "Days must be a whole number between 1 and 3650."
        )

    start = start or date.today()

    selected = people.loc[people["user id"].eq(user_id)]

    if len(selected) != 1:
        raise ValueError("Select one valid, unique user ID.")

    born = selected.iloc[0]["birth_date"]

    dates = [
        start + timedelta(days=offset)
        for offset in range(days + 1)
    ]

    ages = [age_on(born, current_date) for current_date in dates]

    return pd.DataFrame(
        {
            "Date": pd.to_datetime(dates),
            "Age (completed years)": ages,
        }
    )


def main():
    st.set_page_config(
        page_title="People Age Dashboard",
        page_icon="📅",
        layout="wide",
    )

    st.title("People Age Dashboard")
    st.caption(
        "Calculate current and future ages from dates of birth."
    )

    uploaded = st.file_uploader(
        "Optional: upload a people CSV",
        type=["csv"],
    )

    # This works because the notebook saves this code as app.py.
    default_csv = Path(__file__).resolve().with_name("people-100.csv")
    source = uploaded if uploaded is not None else default_csv

    try:
        people = load_people(source)
    except (
        OSError,
        ValueError,
        UnicodeError,
        pd.errors.ParserError,
    ) as exc:
        st.error(f"Could not load the data: {exc}")
        st.info(
            "Place people-100.csv next to app.py, "
            "or upload it using the button above."
        )
        st.stop()

    st.success(f"{len(people)} people loaded.")

    labels = people.set_index("user id")["label"].to_dict()

    selected_id = st.selectbox(
        "Select a person",
        options=list(labels),
        format_func=lambda user_id: labels[user_id],
    )

    days = st.number_input(
        "Days ahead",
        min_value=1,
        max_value=3650,
        value=30,
        step=1,
    )

    today = date.today()

    try:
        result = make_forecast(
            people,
            selected_id,
            days=int(days),
            start=today,
        )
    except (ValueError, OverflowError) as exc:
        st.error(str(exc))
        st.stop()

    selected_person = people.loc[
        people["user id"].eq(selected_id)
    ].iloc[0]

    st.write(
        "Date of birth:",
        selected_person["birth_date"].isoformat(),
    )

    left, right = st.columns(2)

    left.metric(
        "Current age",
        f"{int(result.iloc[0, 1])} years",
    )

    right.metric(
        f"Age in {int(days)} days",
        f"{int(result.iloc[-1, 1])} years",
    )

    st.caption(
        f"As of {today.isoformat()} (server date). "
        f"Includes today plus {int(days)} future days."
    )

    st.subheader("Age chart")
    st.line_chart(result.set_index("Date"))

    st.caption(
        "A flat line is normal: completed age changes only "
        "on a birthday. For February 29 births, this app uses "
        "March 1 in non-leap years."
    )

    st.subheader("Age table")
    st.dataframe(result, hide_index=True)

    st.download_button(
        label="Download age projection",
        data=result.to_csv(index=False).encode("utf-8"),
        file_name="age-projection.csv",
        mime="text/csv",
    )


if __name__ == "__main__":
    main()

Overwriting app.py


In [5]:
import socket
import subprocess
import sys
import time
import urllib.error
import urllib.request
from pathlib import Path

PORT = 8501
folder = Path.cwd()
app_path = folder / "app.py"
log_path = folder / "streamlit.log"
url = f"http://127.0.0.1:{PORT}"

if not app_path.is_file():
    raise FileNotFoundError(
        "app.py was not found. Run Cell 2 first."
    )

already_running = (
    "dashboard_process" in globals()
    and dashboard_process.poll() is None
)

if already_running:
    print("The dashboard process is already running.")
    print(f"Open: {url}")

else:
    # Check the port before starting another server.
    try:
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as probe:
            probe.bind(("127.0.0.1", PORT))
    except OSError as exc:
        raise RuntimeError(
            f"Port {PORT} is already in use. "
            "Stop the existing app or change PORT above."
        ) from exc

    with log_path.open("w", encoding="utf-8") as log:
        dashboard_process = subprocess.Popen(
            [
                sys.executable,
                "-m",
                "streamlit",
                "run",
                str(app_path),
                "--server.address=127.0.0.1",
                f"--server.port={PORT}",
                "--server.headless=true",
                "--browser.gatherUsageStats=false",
            ],
            cwd=str(folder),
            stdout=log,
            stderr=subprocess.STDOUT,
        )

    # Bypass proxy settings for this local health check.
    opener = urllib.request.build_opener(
        urllib.request.ProxyHandler({})
    )

    ready = False

    for attempt in range(30):
        if dashboard_process.poll() is not None:
            break

        try:
            with opener.open(
                f"{url}/_stcore/health",
                timeout=1,
            ) as response:
                ready = response.status == 200
        except (urllib.error.URLError, OSError):
            pass

        if ready:
            break

        time.sleep(0.5)

    if ready:
        print("Dashboard started successfully.")
        print(f"Open: {url}")

        if not (folder / "people-100.csv").is_file():
            print(
                "people-100.csv is not in this folder. "
                "Upload it through the dashboard."
            )
    else:
        print("Startup could not be confirmed. Server log:")
        print(
            log_path.read_text(
                encoding="utf-8",
                errors="replace",
            )[-6000:]
        )

Dashboard started successfully.
Open: http://127.0.0.1:8501
